# PCA From Scratch
## Principal Component Analysis — A Mathematical Walkthrough

> *Implementing PCA step-by-step using only NumPy, Matplotlib, and Pandas.*

---

### What you will learn

| Step | Concept |
|------|----------------------------------------------------------|
| 1    | Why dimensionality reduction matters |
| 2    | Standardisation: levelling the playing field |
| 3    | Covariance matrix: capturing relationships between features |
| 4    | Eigenvalues & eigenvectors: finding principal directions |
| 5    | Projection: compressing data into fewer dimensions |
| 6    | Reconstruction: measuring information loss |
| 7    | Visualisation: scree plots, loadings, 2-D scatter |

---

## 0. Setup


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # project root

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

from src import (
    PCAFromScratch,
    standardize, covariance_matrix, sort_eigenpairs,
    reconstruction_error, make_synthetic_dataset,
    plot_before_after,
)

print('NumPy:', np.__version__)
print('All imports OK ✓')

---

## 1. Why Dimensionality Reduction?

Real-world datasets often have **tens or hundreds of features**.  Many of these features are:
- **Correlated** with each other (e.g. height & shoe size)
- **Redundant** (carry the same information)
- Full of **noise** that obscures signal

High dimensionality hurts us in several ways:

| Problem | Effect |
|---------|--------|
| Curse of dimensionality | Distance metrics break down; nearest-neighbour fails |
| Visualisation | Impossible to plot beyond 3 dimensions |
| Computation | Training time grows fast with feature count |
| Over-fitting | Too many features relative to samples |

**PCA** finds a new coordinate system where:
- The first axis points in the direction of **maximum variance**
- The second axis is orthogonal and captures the **next most variance**
- … and so on

We can then drop the low-variance axes with minimal information loss.


---

## 2. Create a Dataset

We will use a synthetic 5-feature dataset with only 2 *real* dimensions of signal.
PCA should recover the 2-D structure.

In [ ]:
X, labels = make_synthetic_dataset(
    n_samples=240,
    n_features=5,
    n_informative=2,
    n_classes=3,
    random_state=42,
)
n, p = X.shape
print(f'Dataset shape: {n} samples × {p} features')
print(f'Class counts : {dict(zip(*np.unique(labels, return_counts=True)))}')

---

## 3. Step 1 — Standardise

PCA is sensitive to scale.  Feature `A` measured in kilometres (range 0–1000) would dominate feature `B` in millimetres (0–1).  **Z-score standardisation** removes this bias:

$$x_{\text{std}} = \frac{x - \mu}{\sigma}$$

After standardisation every feature has **mean 0** and **standard deviation 1**.

In [ ]:
X_std, mu, sigma = standardize(X)

print('Before standardisation:')
print(f'  means = {X.mean(axis=0).round(2)}')
print(f'  stds  = {X.std(axis=0, ddof=1).round(2)}')

print('\nAfter standardisation:')
print(f'  means = {X_std.mean(axis=0).round(10)}')
print(f'  stds  = {X_std.std(axis=0, ddof=1).round(3)}')

---

## 4. Step 2 — Covariance Matrix

The **covariance matrix** $C$ encodes pairwise relationships between features:

$$C = \frac{1}{n-1} X_{\text{std}}^{\top} X_{\text{std}} \qquad \text{shape: } (p \times p)$$

- $C[i, j] > 0$ → features $i$ and $j$ increase together
- $C[i, j] < 0$ → when one increases, the other decreases
- $C[i, i]$ → variance of feature $i$ (= 1 for standardised data)

In [ ]:
C = covariance_matrix(X_std)
print(f'Covariance matrix shape: {C.shape}')
print('\nCovariance matrix C:')
print(np.round(C, 3))

# Visualise as a heatmap
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1)
for i in range(p):
    for j in range(p):
        ax.text(j, i, f'{C[i,j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_xticks(range(p)); ax.set_yticks(range(p))
ax.set_xticklabels([f'F{k+1}' for k in range(p)])
ax.set_yticklabels([f'F{k+1}' for k in range(p)])
ax.set_title('Covariance Matrix', fontweight='bold')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

---

## 5. Step 3 — Eigen-decomposition

We decompose the covariance matrix:

$$C = V \Lambda V^{\top}$$

where:
- $\Lambda$ is a diagonal matrix of **eigenvalues** $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_p$
- $V$ is the matrix of **eigenvectors** (one per column)

**Intuition:** each eigenvector is a *direction* in feature space; its eigenvalue tells you how much variance lies along that direction.

The first eigenvector $v_1$ (largest eigenvalue) is the direction of maximum variance — the **first principal component**.

In [ ]:
# Raw eigen-decomposition
eigenvalues_raw, eigenvectors_raw = np.linalg.eig(C)
eigenvalues_raw = eigenvalues_raw.real
eigenvectors_raw = eigenvectors_raw.real

print('Eigenvalues (unsorted):')
print(np.round(eigenvalues_raw, 4))

# Sort descending
eigenvalues, eigenvectors = sort_eigenpairs(eigenvalues_raw, eigenvectors_raw)
print('\nEigenvalues (sorted descending):')
print(np.round(eigenvalues, 4))

print('\nEigenvectors (columns, sorted):')
print(np.round(eigenvectors, 3))

---

## 6. Step 4 — Explained Variance

Each eigenvalue $\lambda_k$ equals the **variance** of the data along the $k$-th principal component.  The fraction explained by component $k$ is:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}$$

In [ ]:
total_var = eigenvalues.sum()
evr = eigenvalues / total_var
cum_evr = np.cumsum(evr)

print(f'{'PC':>4}  {'Eigenvalue':>12}  {'Var %':>8}  {'Cumul %':>9}')
print('-' * 40)
for i, (ev, r, c) in enumerate(zip(eigenvalues, evr, cum_evr), 1):
    print(f'PC{i:>2}  {ev:>12.4f}  {r*100:>7.2f}%  {c*100:>8.2f}%')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x = np.arange(1, p+1)
axes[0].bar(x, evr * 100, color=['#E03E3E']*2 + ['#a8c4e0']*(p-2))
axes[0].set_title('Explained Variance per Component', fontweight='bold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'PC{i}' for i in x])
axes[0].grid(axis='y', alpha=0.3)

axes[1].plot(x, cum_evr * 100, marker='o', color='#4C72B0', linewidth=2)
axes[1].axhline(90, color='grey', linestyle='--', alpha=0.6, label='90 %')
axes[1].axhline(95, color='grey', linestyle=':', alpha=0.6, label='95 %')
axes[1].axvline(2, color='#E03E3E', linestyle=':', linewidth=1.5, label='k=2')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'PC{i}' for i in x])
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nFirst 2 PCs explain {cum_evr[1]*100:.1f}% of total variance.')

---

## 7. Step 5 — Projection

We select the top $k$ eigenvectors (columns of $V_k$, shape $p \times k$) and project:

$$X_{\text{proj}} = X_{\text{std}} \cdot V_k \qquad \text{shape: } (n \times k)$$

Each row of $X_{\text{proj}}$ is a sample's coordinates in the new $k$-dimensional space.

In [ ]:
k = 2
V_k = eigenvectors[:, :k]     # top-k eigenvectors as columns
X_proj = X_std @ V_k          # shape (n, k)

print(f'Original shape : {X.shape}')
print(f'Projected shape: {X_proj.shape}')
print(f'Data compressed from {p} → {k} dimensions!')

# Plot
fig, ax = plt.subplots(figsize=(7, 5))
classes = sorted(set(labels))
colours = ['#4C72B0', '#DD8452', '#55A868']
for cls, col in zip(classes, colours):
    mask = labels == cls
    ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
               c=col, label=f'Class {cls}', alpha=0.75, s=40,
               edgecolors='white', linewidths=0.5)
ax.set_xlabel(f'PC 1  ({evr[0]*100:.1f} %)', fontsize=11)
ax.set_ylabel(f'PC 2  ({evr[1]*100:.1f} %)', fontsize=11)
ax.set_title('PCA Projection (2D)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

---

## 8. Step 6 — Reconstruction & Error

We can approximately recover the original data by reversing the projection:

$$\hat{X}_{\text{std}} = X_{\text{proj}} \cdot V_k^{\top}$$
$$\hat{X} = \hat{X}_{\text{std}} \cdot \sigma + \mu$$

The **reconstruction error** measures how much information we lost:

$$\text{MSE} = \frac{1}{N} \sum_i \|x_i - \hat{x}_i\|^2$$

In [ ]:
# Reconstruct
X_rec_std = X_proj @ V_k.T             # back to p-dimensional standardised space
X_rec = X_rec_std * sigma + mu         # un-standardise

mse = reconstruction_error(X, X_rec)
print(f'Reconstruction MSE (k={k}): {mse:.4f}')

# Compare for different k
print(f'\n{'k':>3}  {'MSE':>10}  {'Cumul Var %':>12}')
print('-' * 30)
for kk in range(1, p+1):
    Vk = eigenvectors[:, :kk]
    Xp = X_std @ Vk
    Xr = (Xp @ Vk.T) * sigma + mu
    mse_k = reconstruction_error(X, Xr)
    cum_k = cum_evr[kk-1] * 100
    print(f'{kk:>3}  {mse_k:>10.4f}  {cum_k:>11.1f}%')

---

## 9. Using the `PCAFromScratch` Class

All of the above is wrapped in a clean scikit-learn-style class:

In [ ]:
# ── Full pipeline in 4 lines ──────────────────────────────────────────────────
pca = PCAFromScratch(n_components=2)
X_proj_class = pca.fit_transform(X)

pca.explained_variance()

print(f'\nReconstruction MSE : {pca.reconstruction_error(X):.4f}')
print(f'PCA object         : {pca}')

In [ ]:
# ── Visualise with built-in methods ──────────────────────────────────────────
fig = pca.plot_full_dashboard(X, labels=labels, title='Synthetic Dataset — PCA Dashboard')
plt.show()

---

## 10. Auto Component Selection

Pass a float to `n_components` to automatically select enough components to reach a variance threshold:

In [ ]:
for threshold in [0.80, 0.90, 0.95]:
    pca_auto = PCAFromScratch(n_components=threshold)
    pca_auto.fit(X)
    k_chosen = pca_auto.components_.shape[0]
    var_achieved = pca_auto.cumulative_variance_ratio_[-1] * 100
    print(f'threshold={threshold:.0%}  →  k={k_chosen}  (achieves {var_achieved:.1f}% variance)')

---

## 11. Eigenvector Loadings

The loading matrix tells us **how much each original feature contributes to each PC**.
Large absolute values → that feature is important for that component.

In [ ]:
feature_names = [f'Feature {i+1}' for i in range(p)]
fig = pca.plot_eigenvectors(feature_names=feature_names, figsize=(9, 3))
plt.show()

---

## 12. Whitening

**Whitening** scales each PC by $1/\sqrt{\lambda_k}$ so every component has **unit variance**.
This is useful when feeding PCA output into algorithms sensitive to scale (e.g. k-means, ICA).

In [ ]:
pca_white = PCAFromScratch(n_components=2, whiten=True)
X_white = pca_white.fit_transform(X)

print('Without whitening — PC variances:')
print(np.round(X_proj_class.var(axis=0, ddof=1), 3))

print('\nWith whitening — PC variances:')
print(np.round(X_white.var(axis=0, ddof=1), 3))

---

## 13. Summary

```
PCA Pipeline (from scratch)
══════════════════════════════════════════════
  Raw data X  (n × p)
       │
       ▼  Standardise: X_std = (X - μ) / σ
  X_std  (n × p)
       │
       ▼  Covariance: C = X_std.T @ X_std / (n-1)
  C  (p × p)
       │
       ▼  Eigen-decompose: C = V Λ Vᵀ
  λ₁≥λ₂≥…≥λₚ,  V (p × p)
       │
       ▼  Select top k
  V_k  (p × k)
       │
       ▼  Project: X_proj = X_std @ V_k
  X_proj  (n × k)
       │
       ▼  Reconstruct: X̂ = X_proj @ V_kᵀ · σ + μ
  X̂  (n × p)  +  MSE error metric
══════════════════════════════════════════════
```

### References

- Jolliffe, I.T. (2002). *Principal Component Analysis*, 2nd ed. Springer.
- Shlens, J. (2014). [A Tutorial on Principal Component Analysis](https://arxiv.org/abs/1404.1100). arXiv.
- NumPy docs: [numpy.linalg.eig](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html)
